In [1]:
!pip install groq PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 10.4 MB/s eta 0:00:00


In [7]:
from google.colab import files

print("Click 'Choose Files' and select your PDF")
uploaded = files.upload()

PDF_PATH = list(uploaded.keys())[0]
print(f"✅ Uploaded: {PDF_PATH}")

Click 'Choose Files' and select your PDF


Saving artificial_intelligence.pdf to artificial_intelligence.pdf
✅ Uploaded: artificial_intelligence.pdf


In [6]:
from groq import Groq

API_KEY = "xxxxx"   # ← get free key at console.groq.com
MODEL      = "llama-3.3-70b-versatile"
CHUNK_SIZE = 3000
OUTPUT     = "summary.txt"

client = Groq(api_key=API_KEY)
print("✅ Connected to Groq AI!")

✅ Connected to Groq AI!


In [8]:
import PyPDF2

def read_pdf(pdf_path):
    print(f"📄 Reading: {pdf_path}")
    all_text = ""

    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        total = len(reader.pages)
        print(f"   Found {total} pages")

        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text:
                all_text += text + "\n"

    print(f"✅ Extracted {len(all_text)} characters")
    return all_text

# Run it
text = read_pdf(PDF_PATH)

📄 Reading: artificial_intelligence.pdf
   Found 2 pages
✅ Extracted 3576 characters


In [9]:
def split_into_chunks(text, chunk_size=CHUNK_SIZE):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        # Try to end at a sentence
        if end < len(text):
            period = text.rfind(". ", start, end)
            if period != -1:
                end = period + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end

    print(f"✅ Split into {len(chunks)} chunks")
    return chunks

# Run it
chunks = split_into_chunks(text)

✅ Split into 2 chunks


In [10]:
def summarize_chunk(chunk, num, total):
    print(f"   Summarizing chunk {num}/{total}...")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": f"Summarize this text clearly and concisely:\n\n{chunk}"
        }],
        max_tokens=500,
        temperature=0.3
    )
    return response.choices[0].message.content

# Run it
print(f"🤖 Summarizing {len(chunks)} chunks...")
chunk_summaries = []

for i, chunk in enumerate(chunks, start=1):
    summary = summarize_chunk(chunk, i, len(chunks))
    chunk_summaries.append(summary)

print("✅ All chunks summarized!")

🤖 Summarizing 2 chunks...
   Summarizing chunk 1/2...
   Summarizing chunk 2/2...
✅ All chunks summarized!


In [11]:
# ── Step 1: Choose language ───────────────────────────────────
languages = {
    "1":  "Hindi",
    "2":  "Spanish",
    "3":  "French",
    "4":  "Arabic",
    "5":  "German",
    "6":  "Chinese (Simplified)",
    "7":  "Japanese",
    "8":  "Portuguese",
    "9":  "Russian",
    "10": "Bengali",
    "11": "Urdu",
    "12": "English"
}

print("=" * 50)
print("Choose your output language:")
print("=" * 50)
for key, lang in languages.items():
    print(f"  {key:>2} = {lang}")
print("=" * 50)

lang_choice = input("Enter number: ").strip()

if lang_choice not in languages:
    print("Invalid choice — defaulting to English")
    lang_choice = "12"

chosen_language = languages[lang_choice]
print(f"✅ Language: {chosen_language}")

# ── Step 2: Choose summary length ────────────────────────────
print("\n" + "=" * 50)
print("Choose your summary length:")
print("=" * 50)
print("  1 = Short     (100-150 words)")
print("  2 = Medium    (250-300 words)")
print("  3 = Detailed  (500+ words)")
print("=" * 50)

length_choice = input("Enter 1, 2, or 3: ").strip()

length_map = {
    "1": {
        "label":       "Short",
        "instruction": "Write a SHORT summary in 100-150 words. Only the most important points.",
        "max_tokens":  250
    },
    "2": {
        "label":       "Medium",
        "instruction": "Write a MEDIUM summary in 250-300 words. Cover all key points clearly.",
        "max_tokens":  450
    },
    "3": {
        "label":       "Detailed",
        "instruction": "Write a DETAILED summary in 500+ words. Cover everything thoroughly.",
        "max_tokens":  800
    }
}

if length_choice not in length_map:
    print("Invalid choice — defaulting to Medium")
    length_choice = "2"

length_settings = length_map[length_choice]
print(f"✅ Length: {length_settings['label']}")


# ── Step 3: Summarize each chunk directly in chosen language ──
def summarize_chunk(chunk, num, total):
    print(f"   Summarizing chunk {num}/{total} in {chosen_language}...")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": f"""{length_settings['instruction']}
Write the summary ONLY in {chosen_language} language.
Do not use English unless {chosen_language} is English.

Text:
{chunk}"""
        }],
        max_tokens=length_settings["max_tokens"],
        temperature=0.3
    )
    return response.choices[0].message.content

# Run it
print(f"\n🤖 Summarizing {len(chunks)} chunks in {chosen_language} ({length_settings['label']})...")
chunk_summaries = []

for i, chunk in enumerate(chunks, start=1):
    summary = summarize_chunk(chunk, i, len(chunks))
    chunk_summaries.append(summary)

print("✅ Done!")

Choose your output language:
   1 = Hindi
   2 = Spanish
   3 = French
   4 = Arabic
   5 = German
   6 = Chinese (Simplified)
   7 = Japanese
   8 = Portuguese
   9 = Russian
  10 = Bengali
  11 = Urdu
  12 = English
Enter number: 9
✅ Language: Russian

Choose your summary length:
  1 = Short     (100-150 words)
  2 = Medium    (250-300 words)
  3 = Detailed  (500+ words)
Enter 1, 2, or 3: 2
✅ Length: Medium

🤖 Summarizing 2 chunks in Russian (Medium)...
   Summarizing chunk 1/2 in Russian...
   Summarizing chunk 2/2 in Russian...
✅ Done!


In [12]:
def final_summary(summaries):
    print("🔗 Creating final summary...")

    combined = "\n\n".join(summaries)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": f"""Combine these into ONE clear final summary.
Use simple language. Keep it under 500 words.

{combined}"""
        }],
        max_tokens=800,
        temperature=0.3
    )
    return response.choices[0].message.content

# Run it
if len(chunk_summaries) == 1:
    result = chunk_summaries[0]
else:
    result = final_summary(chunk_summaries)

print("\n" + "="*50)
print("FINAL SUMMARY:")
print("="*50)
print(result)

🔗 Creating final summary...

FINAL SUMMARY:
**Что такое искусственный интеллект?**

Искусственный интеллект (ИИ) - это технология, которая позволяет машинам выполнять задачи, обычно требующие человеческого интеллекта. Это включает обучение на опыте, рассуждение, решение проблем и понимание естественного языка. ИИ работает путем объединения больших объемов данных с интеллектуальными алгоритмами, которые позволяют машинам учиться и улучшать свою производительность со временем.

**Типы ИИ**

Существует три основных типа ИИ: узкий ИИ (слабый ИИ), общий ИИ и супер ИИ. Узкий ИИ предназначен для выполнения конкретных задач, таких как распознавание голоса или классификация изображений. Общий ИИ - это теоретическая концепция, где машины будут иметь возможность выполнять любую интеллектуальную задачу, которую может выполнить человек. Супер ИИ - это будущая концепция, где машины превосходят человеческий интеллект во всех аспектах.

**Применения и преимущества ИИ**

ИИ имеет много реальных примене

In [13]:
def analyze_sentiment(text):
    print("\n🔍 Analyzing sentiment of the document...")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": f"""Analyze the sentiment and tone of this document.

Return your analysis in EXACTLY this format, nothing else:

OVERALL: [Positive / Negative / Neutral / Mixed]
SCORE: [a number from 1 to 10, where 1=very negative, 5=neutral, 10=very positive]
TONE: [2-3 words describing the tone, e.g. "formal and informative"]
EMOTIONS: [list 3 emotions found, e.g. "optimism, concern, curiosity"]
REASON: [one sentence explaining why you gave this sentiment]

Document:
{text[:4000]}"""
        }],
        max_tokens=200,
        temperature=0.2
    )

    return response.choices[0].message.content

# Run it
sentiment_raw = analyze_sentiment(text)

# Parse the result line by line
sentiment = {}
for line in sentiment_raw.strip().split("\n"):
    if ":" in line:
        key, val = line.split(":", 1)
        sentiment[key.strip()] = val.strip()

# Display nicely
print("\n" + "="*50)
print("SENTIMENT ANALYSIS REPORT")
print("="*50)

overall = sentiment.get("OVERALL", "Unknown")
score   = sentiment.get("SCORE", "?")
tone    = sentiment.get("TONE", "Unknown")
emotions = sentiment.get("EMOTIONS", "Unknown")
reason  = sentiment.get("REASON", "")

# Sentiment emoji based on result
emoji_map = {
    "Positive": "😊",
    "Negative": "😟",
    "Neutral":  "😐",
    "Mixed":    "🤔"
}
emoji = emoji_map.get(overall, "📊")

print(f"\n{emoji}  Overall Sentiment : {overall}")
print(f"📊  Score (1-10)      : {score}")
print(f"🎭  Tone              : {tone}")
print(f"💭  Emotions detected : {emotions}")
print(f"💡  Reason            : {reason}")

# Visual score bar
try:
    score_num = int(score.split("/")[0].strip())
    filled    = int((score_num / 10) * 20)
    bar       = "█" * filled + "░" * (20 - filled)
    print(f"\n     [{bar}] {score_num}/10")
except:
    pass

print("="*50)


🔍 Analyzing sentiment of the document...

SENTIMENT ANALYSIS REPORT

😊  Overall Sentiment : Positive
📊  Score (1-10)      : 8
🎭  Tone              : Informative and objective
💭  Emotions detected : Curiosity, optimism, excitement
💡  Reason            : The document provides a comprehensive and balanced explanation of Artificial Intelligence, highlighting its benefits and potential, while also acknowledging challenges, resulting in an overall positive sentiment.

     [████████████████░░░░] 8/10


In [14]:
from google.colab import files

# Ask for filename
print("=" * 50)
custom_name = input("Enter a name for your summary file (without .txt): ")
if not custom_name.strip():
    custom_name = "summary"
OUTPUT = custom_name.strip().replace(" ", "_") + ".txt"

# Save everything together
with open(OUTPUT, "w", encoding="utf-8") as f:

    # ── Header ───────────────────────────────────────
    f.write("=" * 50 + "\n")
    f.write("       AI DOCUMENT SUMMARY REPORT\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"File    : {PDF_PATH}\n")
    f.write(f"Model   : {MODEL}\n")
    f.write(f"Length  : {length_settings['label']}\n")   # Short / Medium / Detailed
    f.write("\n" + "=" * 50 + "\n\n")

    # ── Summary section ──────────────────────────────
    f.write(f"SUMMARY  ({length_settings['label'].upper()})\n")
    f.write("-" * 50 + "\n")
    f.write(result + "\n\n")

    # ── Sentiment section ────────────────────────────
    f.write("=" * 50 + "\n")
    f.write("SENTIMENT ANALYSIS\n")
    f.write("-" * 50 + "\n")

    try:
        # Score bar
        score_num = int(score.split("/")[0].strip())
        filled    = int((score_num / 10) * 20)
        bar       = "█" * filled + "░" * (20 - filled)

        f.write(f"Overall  : {overall}\n")
        f.write(f"Score    : [{bar}] {score_num}/10\n")
        f.write(f"Tone     : {tone}\n")
        f.write(f"Emotions : {emotions}\n")
        f.write(f"Reason   : {reason}\n")
    except:
        f.write("Sentiment analysis was not run.\n")
        f.write("Run the sentiment cell before saving to include it.\n")

    f.write("\n" + "=" * 50 + "\n")
    f.write("         END OF REPORT\n")
    f.write("=" * 50 + "\n")

print(f"✅ Saved: {OUTPUT}")
files.download(OUTPUT)
print("🎉 Done! Check your downloads folder.")

Enter a name for your summary file (without .txt): russian
✅ Saved: russian.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 Done! Check your downloads folder.
